# Load Relevant modules

In [ ]:
from mace.calculators import MACECalculator
from ase.build.surface import mx2
import numpy as np
import os
import matplotlib.pyplot as plt
from ase.io.espresso import read_espresso_ph
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.ticker import MultipleLocator
from ase import units
from ase.io import read, Trajectory
from glob import glob
from sklearn.metrics import mean_squared_error

# Alloys

## Load Alloys Dielectrics Model for testing

In [ ]:
calc_raman  = MACECalculator(model_paths="../training/alloys_dielectric.model",
                             model_type="DipoleMACE",device="cuda", default_dtype="float64")

## Store Data for Plotting

In [ ]:
all_files = glob(f'alloys_test_data/*.ph.out')

#Arrays for storing data
dft_suscs = np.zeros((len(all_files),3,3))
mace_suscs = np.zeros((len(all_files),3,3))
dft_ramans = np.zeros((len(all_files),len(atoms)*3,3,3))
mace_ramans = np.zeros((len(all_files),len(atoms)*3,3,3))

for i,file in enumerate(all_files):

    #Get DFT data
    ph = read_espresso_ph(open(file))[1]
    atoms = ph['atoms']
    dft_susc = -(ph['dieltensor']-np.identity(3))*(atoms.cell.volume/(4*np.pi))#
    dft_suscs[i] = dft_susc
    dft_raman = np.array(ph['raman_tensor']).reshape(len(atoms)*3,3,3)
    dft_ramans[i] = dft_raman

    #Get MACE data on configuration
    calc_raman.calculate(atoms)
    mace_susc = calc_raman.results['polarizability']
    mace_suscs[i] = mace_susc
    mace_raman = calc_raman.results['polarizability_deriv']
    mace_raman = -mace_raman.reshape(len(atoms),3,3,3).transpose((0, 3, 1, 2)).reshape(len(atoms)*3,3,3)
    mace_ramans[i] = mace_raman

## Plot DFT vs ML 

In [ ]:
fontsize = 13
plt.rcParams['font.size'] = fontsize
fig, axs = plt.subplots(2, 3, figsize=(7.5, 5))

# Define labels for subplots
labels = [r'$\chi_{xx}$', r'$\chi_{yy}$', r'$\chi_{zz}$', r'$\chi_{xy}$', r'$\chi_{yz}$', r'$\chi_{xz}$']
ax_indices = [[0,0],[1,1],[2,2],[0,1],[1,2],[0,2]]

# Loop through each subplot and plot data
for i, ax in enumerate(axs.flat):

    x_data = dft_suscs[:, ax_indices[i][0], ax_indices[i][1]]
    y_data = mace_suscs[:, ax_indices[i][0], ax_indices[i][1]]

    ax.scatter(x_data, y_data,s=1)

    # Compute RMSE
    rmse = np.sqrt(mean_squared_error(x_data, y_data))

    # Add x=y reference line
    min_val, max_val = min(x_data.min(), y_data.min()), max(x_data.max(), y_data.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.7)

    # Add text labels
    ax.text(0.2, 0.95, labels[i], transform=ax.transAxes, ha='center', va='top', fontsize=fontsize+5)
    ax.text(0.97, 0.05, f'RMSE/atom:\n{rmse/108:.3f}', transform=ax.transAxes, ha='right', va='bottom', fontsize=fontsize)

# Set axis labels for the whole figure
fig.text(0.5, -0.01, r"$\chi_{DFT}$ (a.u.)", ha='center', fontsize=fontsize+5)
fig.text(-0.01, 0.5, r"$\chi_{ML}$ (a.u.)", va='center', rotation='vertical', fontsize=fontsize+5)

plt.tight_layout()
plt.subplots_adjust(wspace=0.45, hspace=0.2)
plt.savefig('alloys_pol_test.png',dpi=300,facecolor='white',transparent=False,
            bbox_inches='tight',pad_inches=0.02)
plt.show()

## MD Test

In [ ]:
## MD Alloy composition and temperature
x,y,temp = 0.5,0.5,300

ts = np.arange(0,80,5)
dft_suscs = np.zeros((len(ts),3,3))
mace_suscs = np.zeros((len(ts),3,3))

for i,t in enumerate(ts):
    
    filename = f'alloys_md_test_data/x{x:.3f}y{y:.3f}T{temp:04.0f}fs{t:02d}.ph.out'
    
    ph = read_espresso_ph(open(filename))[1]
    atoms = ph['atoms']
    dft_susc = -(ph['dieltensor']-np.identity(3))*(atoms.cell.volume/(4*np.pi))
    dft_suscs[i] = dft_susc
    calc_raman.calculate(atoms)
    mace_susc = calc_raman.results['polarizability']
    mace_suscs[i] = mace_susc

## Plot DFT vs ML MD

In [ ]:
fontsize = 12
markersize = 3  # <--- Change this to control all marker sizes

plt.rcParams['font.size'] = fontsize
fig, axs = plt.subplots(2, 3, figsize=(7.5, 5), sharex='col')

axs[0][0].plot(ts, dft_suscs[:,0,0], 'o--', label='DFT', markersize=markersize)
axs[0][0].plot(ts, mace_suscs[:,0,0], 'o--', label='ML', markersize=markersize)
axs[0][0].text(0.5, 0.95, r'$\chi_{xx}$', transform=axs[0][0].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[0][0].set_ylim(-2490, -2440)

handles, labels = axs[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', fontsize=fontsize,
           ncol=len(labels), frameon=False, bbox_to_anchor=(0.5, 0.95),
           prop={'weight': 'bold'}, columnspacing=2)

axs[0][1].plot(ts, dft_suscs[:,1,1], 'o--', label='DFT', markersize=markersize)
axs[0][1].plot(ts, mace_suscs[:,1,1], 'o--', label='ML', markersize=markersize)
axs[0][1].text(0.5, 0.95, r'$\chi_{yy}$', transform=axs[0][1].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[0][1].set_ylim(-2490, -2440)

axs[0][2].plot(ts, dft_suscs[:,2,2], 'o--', label='DFT', markersize=markersize)
axs[0][2].plot(ts, mace_suscs[:,2,2], 'o--', label='ML', markersize=markersize)
axs[0][2].text(0.5, 0.95, r'$\chi_{zz}$', transform=axs[0][2].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[0][2].set_ylim(-170, -120)

axs[1][0].plot(ts, dft_suscs[:,0,1], 'o--', label='DFT', markersize=markersize)
axs[1][0].plot(ts, mace_suscs[:,0,1], 'o--', label='ML', markersize=markersize)
axs[1][0].text(0.5, 0.95, r'$\chi_{xy}$', transform=axs[1][0].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[1][0].set_ylim(-5, 5)

axs[1][1].plot(ts, dft_suscs[:,1,2], 'o--', label='DFT', markersize=markersize)
axs[1][1].plot(ts, mace_suscs[:,1,2], 'o--', label='ML', markersize=markersize)
axs[1][1].text(0.5, 0.95, r'$\chi_{yz}$', transform=axs[1][1].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[1][1].set_ylim(-5, 5)

axs[1][2].plot(ts, dft_suscs[:,0,2], 'o--', label='DFT', markersize=markersize)
axs[1][2].plot(ts, mace_suscs[:,0,2], 'o--', label='ML', markersize=markersize)
axs[1][2].text(0.5, 0.95, r'$\chi_{xz}$', transform=axs[1][2].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[1][2].set_ylim(-5, 5)

for i, ax in enumerate(axs.flat):
    ax.xaxis.set_major_locator(MultipleLocator(20))
    ax.xaxis.set_minor_locator(MultipleLocator(5))
    if i < 3:
        ax.yaxis.set_minor_locator(MultipleLocator(5))
    else:
        ax.yaxis.set_minor_locator(MultipleLocator(1))

fig.text(0.5, 0.02, r"Time (in fs)", ha='center', fontsize=fontsize+5)
fig.text(0.001, 0.5, r"$\chi$ (a.u.)", va='center', rotation='vertical', fontsize=fontsize+5)
fig.gca().xaxis.set_minor_locator(MultipleLocator(5))
plt.subplots_adjust(wspace=0.45, hspace=0.05)

plt.savefig('alloys_md_traj_test.png', dpi=300, facecolor='white', transparent=False,
            bbox_inches='tight', pad_inches=0.02)

plt.show()


# Aligned Bilayers

## Load aligned bilayers dielectrics model

In [ ]:
calc_raman  = MACECalculator(model_paths="../training/aligned_bilayer_dielectrics.model",
                             model_type="DipoleMACE",device="cuda", default_dtype="float64")

## Store Data for plotting

In [ ]:
all_files = glob(f'aligned_test_data/*.ph.out')
#all_files = glob(f'trilayer_ph/*.ph.out')

dft_suscs = np.zeros((len(all_files),3,3))
mace_suscs = np.zeros((len(all_files),3,3))
dft_ramans = np.zeros((len(all_files),len(atoms)*3,3,3))
mace_ramans = np.zeros((len(all_files),len(atoms)*3,3,3))


for i,file in enumerate(all_files):
    ph = read_espresso_ph(open(file))[1]
    atoms = ph['atoms']
    dft_susc = -(ph['dieltensor']-np.identity(3))*(atoms.cell.volume/(4*np.pi))
    dft_suscs[i] = dft_susc
    dft_raman = np.array(ph['raman_tensor']).reshape(len(atoms)*3,3,3)
    dft_ramans[i] = dft_raman
    
    calc_raman.calculate(atoms)
    mace_susc = calc_raman.results['polarizability']
    mace_suscs[i] = mace_susc
    mace_raman = calc_raman.results['polarizability_deriv']
    mace_raman = -mace_raman.reshape(len(atoms),3,3,3).transpose((0, 3, 1, 2)).reshape(len(atoms)*3,3,3)
    mace_ramans[i] = mace_raman

## Plot DFT vs ML

In [ ]:
fontsize = 13
plt.rcParams['font.size'] = fontsize
fig, axs = plt.subplots(2, 3, figsize=(7.5, 5))

# Define labels for subplots

labels = [r'$\chi_{xx}$', r'$\chi_{yy}$', r'$\chi_{zz}$', r'$\chi_{xy}$', r'$\chi_{yz}$', r'$\chi_{xz}$']
ax_indices = [[0,0],[1,1],[2,2],[0,1],[1,2],[0,2]]

# Loop through each subplot and plot data
for i, ax in enumerate(axs.flat):
    
    x_data = dft_suscs[:, ax_indices[i][0], ax_indices[i][1]]
    y_data = mace_suscs[:, ax_indices[i][0], ax_indices[i][1]]

    ax.scatter(x_data, y_data,s=1)

    # Compute RMSE
    rmse = np.sqrt(mean_squared_error(x_data, y_data))

    # Add x=y reference line
    min_val, max_val = min(x_data.min(), y_data.min()), max(x_data.max(), y_data.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.7)

    # Add text labels
    ax.text(0.2, 0.95, labels[i], transform=ax.transAxes, ha='center', va='top', fontsize=fontsize+5)
    ax.text(0.97, 0.05, f'RMSE/atom:\n{rmse/6:.3f}', transform=ax.transAxes, ha='right', va='bottom', fontsize=fontsize)

# Set axis labels for the whole figure
fig.text(0.5, -0.005, r"$\chi_{DFT}$ (a.u.)", ha='center', fontsize=fontsize+5)
fig.text(-0.005, 0.5, r"$\chi_{ML}$ (a.u.)", va='center', rotation='vertical', fontsize=fontsize+5)

plt.tight_layout()
plt.subplots_adjust(wspace=0.32, hspace=0.17)
plt.savefig('aligned_pol_test.png',dpi=300,facecolor='white',transparent=False,
            bbox_inches='tight',pad_inches=0.02)
plt.show()

## MD Test

In [ ]:
m1 = 'Mo'
m2 = 'Mo'
ch = 'S'
stacking = '2H'
temp = 300

ts = np.arange(0,80,5)
dft_suscs = np.zeros((len(ts),3,3))
mace_suscs = np.zeros((len(ts),3,3))

for i,t in enumerate(ts):
    filename = f'md_traj_dft/{m1}{m2}{ch}_{stacking}_T{temp:04.0f}_fs{t:02d}.ph.out'
    ph = read_espresso_ph(open(filename))[1]
    atoms = ph['atoms']
    dft_susc = -(ph['dieltensor']-np.identity(3))*(atoms.cell.volume/(4*np.pi))
    dft_suscs[i] = dft_susc
    calc_raman.calculate(atoms)
    mace_susc = calc_raman.results['polarizability']
    mace_suscs[i] = mace_susc

## Plot DFT vs ML MD

In [ ]:
fontsize = 12
markersize = 3  # <-- You can change this value to adjust marker size

plt.rcParams['font.size'] = fontsize
fig, axs = plt.subplots(2, 3, figsize=(7.5, 5), sharex='col')

axs[0][0].plot(ts, dft_suscs[:,0,0], 'o--', label='DFT', markersize=markersize)
axs[0][0].plot(ts, mace_suscs[:,0,0], 'o--', label='ML', markersize=markersize)
axs[0][0].text(0.5, 0.95, r'$\chi_{xx}$', transform=axs[0][0].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[0][0].set_ylim(-4590,-4510)

handles, labels = axs[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', fontsize=fontsize,
           ncol=len(labels), frameon=False, bbox_to_anchor=(0.5, 0.95),
           prop={'weight': 'bold'}, columnspacing=2)


axs[0][1].plot(ts, dft_suscs[:,1,1], 'o--', label='DFT', markersize=markersize)
axs[0][1].plot(ts, mace_suscs[:,1,1], 'o--', label='ML', markersize=markersize)
axs[0][1].text(0.5, 0.95, r'$\chi_{yy}$', transform=axs[0][1].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[0][1].set_ylim(-4590,-4510)

axs[0][2].plot(ts, dft_suscs[:,2,2], 'o--', label='DFT', markersize=markersize)
axs[0][2].plot(ts, mace_suscs[:,2,2], 'o--', label='ML', markersize=markersize)
axs[0][2].text(0.5, 0.95, r'$\chi_{zz}$', transform=axs[0][2].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[0][2].set_ylim(-700, -500)

axs[1][0].plot(ts, dft_suscs[:,0,1], 'o--', label='DFT', markersize=markersize)
axs[1][0].plot(ts, mace_suscs[:,0,1], 'o--', label='ML', markersize=markersize)
axs[1][0].text(0.5, 0.95, r'$\chi_{xy}$', transform=axs[1][0].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[1][0].set_ylim(-5, 10)

axs[1][1].plot(ts, dft_suscs[:,1,2], 'o--', label='DFT', markersize=markersize)
axs[1][1].plot(ts, mace_suscs[:,1,2], 'o--', label='ML', markersize=markersize)
axs[1][1].text(0.5, 0.95, r'$\chi_{yz}$', transform=axs[1][1].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[1][1].set_ylim(-5, 10)

axs[1][2].plot(ts, dft_suscs[:,0,2], 'o--', label='DFT', markersize=markersize)
axs[1][2].plot(ts, mace_suscs[:,0,2], 'o--', label='ML', markersize=markersize)
axs[1][2].text(0.5, 0.95, r'$\chi_{xz}$', transform=axs[1][2].transAxes,
               ha='center', va='top', fontsize=fontsize+5)
axs[1][2].set_ylim(-5, 10)

for i, ax in enumerate(axs.flat):
    ax.xaxis.set_major_locator(MultipleLocator(20))
    ax.xaxis.set_minor_locator(MultipleLocator(5))
    if i < 3:
        ax.yaxis.set_minor_locator(MultipleLocator(5))
    else:
        ax.yaxis.set_minor_locator(MultipleLocator(0.5))

fig.text(0.5, 0.02, r"Time (in fs)", ha='center', fontsize=fontsize+5)
fig.text(0.001, 0.5, r"$\chi$ (a.u.)", va='center', rotation='vertical', fontsize=fontsize+5)
fig.gca().xaxis.set_minor_locator(MultipleLocator(5))
plt.subplots_adjust(wspace=0.45, hspace=0.15)

plt.savefig('aligned_md_traj_test.png', dpi=300, facecolor='white', transparent=False,
            bbox_inches='tight', pad_inches=0.02)

plt.show()
